# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
import os, sys, math, time, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Essential Dependencies (These are usually missing on Kaggle)
!pip install -q bitsandbytes wandb datasets transformers

# 2. Pull Research Source (Injecting into path instead of installing)
REPO_NAME = 'EFV-nn'
REPO_URL = 'https://github.com/ey3lock3r/EFV-nn.git'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone -q {REPO_URL}')
else:
    os.system(f'cd {REPO_NAME} && git pull -q')

sys.path.append(f'{REPO_NAME}/src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import bitsandbytes as bnb
import wandb
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from kaggle_secrets import UserSecretsClient

from efv_nn.ppc_sharded import ShardedPPCGraphLLM

print("✅ Environment ready. Source loaded from GitHub.")
print(f"CUDA Available: {torch.cuda.is_available()} | Devices: {torch.cuda.device_count()}")


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print("✅ Logged into W&B and HF")
except Exception as e:
    print(f"⚠️ Secrets failure: {e}. Training will continue without W&B.")


In [ ]:
# ── Cell 4: 🏗️ Architectural Setup (Run ONCE) ───────────────────────────
VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
CKPT_PATH = "/kaggle/working/ppc_3b_pretrain.pt"
RESUME = True # Set to False to start fresh from Kaiming initialization

print("📥 Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))

print("🏗️ Instantiating 3.2B Sharded Model (VRAM Resident)...")
gc.collect(); torch.cuda.empty_cache()
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, use_jacobian=True)

START_STEP = 0
if RESUME and os.path.exists(CKPT_PATH):
    print(f"🔄 Loading Checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location='cpu')
    model.load_state_dict(ckpt['model_state'], strict=False)
    START_STEP = ckpt.get('step', 0)
    print(f"✅ Resumed Weight Step {START_STEP}")
else:
    print("✨ Starting with fresh random weights.")

print(f"✅ Model Ready. Total Params: {model.total_params/1e9:.2f}B")

In [ ]:
# ── Cell 5: 🚄 Training Engine ──────────────────
def save_checkpoint(model, step, path):
    gc.collect(); torch.cuda.empty_cache()
    state_dict = model.state_dict()
    compressed_dict = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in state_dict.items()}
    torch.save({'step': step, 'model_state': compressed_dict}, path)
    wandb.save(path)
    print(f"✅ Step {step} saved.")

def train(model, tokenizer, num_steps=1000, start_step=0):
    wandb.init(project="ppc-3b-dynamic", name=f"ppc-3b-{time.strftime('%H%M')}")
    dataloader = get_dataloader(tokenizer, batch_size=2, seq_len=256)
    
    # Paged AdamW8bit offloads states to CPU to save VRAM
    opt = bnb.optim.PagedAdamW8bit(model.parameters(), lr=1.5e-4)
    
    t0 = time.time()
    last_step = start_step
    
    print(f"🚀 Starting training loop from step {start_step}...")
    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        ids, targets = batch[:, :-1], batch[:, 1:].to(model.device1)
        
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            # Resonance monitoring (E) and Island-Fused PPC loop (It)
            logits, avg_iters, avg_energy = model(ids, local_iters=16)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
        
        (loss / 256.0).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0 / 256.0)
        opt.step()
        
        if step % 5 == 0:
            ppl = math.exp(min(loss.item(), 20))
            lr = opt.param_groups[0]['lr']
            dt = (time.time() - t0) * 1000 / (step - last_step) if step > last_step else 0
            t0 = time.time(); last_step = step
            wandb.log({"loss": loss.item(), "ppl": ppl, "avg_iters": avg_iters, "avg_energy": avg_energy, "lr": lr, "duration_ms": dt}, step=step)
            print(f"St {step} | L: {loss.item():.4f} | E: {avg_energy:.4f} | It: {avg_iters:.1f} | D: {dt:.0f}ms | LR: {lr:.2e}")
            
        if (step + 1) % 100 == 0: save_checkpoint(model, step + 1, CKPT_PATH)
        if step >= num_steps: break

# To run: 
# train(model, tokenizer, start_step=START_STEP)

In [ ]:
# ── Cell 6: 🗣️ Verification (Instant) ──────────────────
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    out_reg = model.generate(inputs, max_new_tokens=40)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    out_swarm = model.generate_swarm(inputs, max_new_tokens=40, swarm_size=8)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")

# To run: 
# verify_generation(model, tokenizer)

In [ ]:
# ── Cell 5: 🗣️ Generation Verification ──────────────────
def verify_generation(prompt="The fundamental principles of biology include"):
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))
    model = ShardedPPCGraphLLM(128256, 1024, 24, 64)
    if os.path.exists(CKPT_PATH):
        ckpt = torch.load(CKPT_PATH, map_location='cpu')
        model.load_state_dict(ckpt['model_state'], strict=False)
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    print("\n🧠 Running Regular Generation...")
    output_ids = model.generate(inputs, max_new_tokens=50)
    print(f"Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")
    
    print("\n🐝 Running Advanced Swarm Inference (N=8)...")
    swarm_ids = model.generate_swarm(inputs, max_new_tokens=50, swarm_size=8)
    print(f"Output:\n{tokenizer.decode(swarm_ids[0], skip_special_tokens=True)}")
    print(f"\nPrompt: {prompt}")

verify_generation()
